# BIRD Targeted-SFT Data Construction v1

This notebook turns the completed MiniDev500 failure analysis into a **training-only, leakage-safe Targeted-SFT dataset**.

Fixed design decisions from failure analysis v3:

- MiniDev500 is used only to choose aggregate capability weights. No MiniDev500 row is copied into training.
- The original BIRD train/validation database split is preserved.
- Training output size is held equal to the original SFT training size so optimizer-step comparisons remain controllable.
- 65% of sampled prompts are targeted and 35% are general replay.
- Targeted quotas are 40% schema/relationship, 22% predicate/evidence, 20% computation logic, and 18% result structure.
- The first controlled run preserves each source record's original evidence-dropout condition. Forced evidence on targeted copies remains an explicit ablation switch.
- Sampling is deterministic, database-balanced, complexity-aware, and capped per source example.

Why these weights: SFT + evidence produced 85 schema/relationship, 42 predicate/evidence, 45 computation, and 40 result-structure failures. Its 47 regressions were also concentrated in schema/relationship and predicate/evidence errors. A 35% replay share is retained because the original SFT lost 6.8 percentage points on moderate examples.


## Expected inputs and outputs

Inputs already created by notebook 02:

- `processed/train_sft_v2.jsonl`
- `processed/validation_sft_v2.jsonl`
- `processed/split_metadata_v2.json`

Optional provenance input from failure analysis v3:

- `results/minidev500_failure_mode_analysis_with_evidence_v3/failure_mode_analysis_summary_v3.json`

Outputs are written under `processed/targeted_sft_v1/`:

- `train_targeted_sft_v1.jsonl` — use for training
- `validation_general_sft_v1.jsonl` — use for checkpoint selection; content matches the original general validation set
- `validation_targeted_sft_v1.jsonl` — use as a capability diagnostic, not the sole checkpoint-selection set
- row-level selection indexes, distribution summaries, and a reproducibility manifest


In [1]:
!pip -q install sqlglot==25.20.2


In [2]:
from google.colab import drive

drive.mount("/content/drive")


Mounted at /content/drive


In [3]:
from collections import Counter, defaultdict
from copy import deepcopy
from hashlib import sha256
from pathlib import Path
import json
import math
import os
import random
import re
import tempfile

import numpy as np
import pandas as pd
import sqlglot
from sqlglot import exp

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("sqlglot:", sqlglot.__version__)


pandas: 2.2.2
numpy: 2.0.2
sqlglot: 25.20.2


In [4]:
# ------------------------- Project paths -------------------------
PROJECT_ROOT = Path("/content/drive/MyDrive/bird-text2sql-sft")
PROCESSED_DIR = PROJECT_ROOT / "processed"

TRAIN_INPUT_PATH = PROCESSED_DIR / "train_sft_v2.jsonl"
VALIDATION_INPUT_PATH = PROCESSED_DIR / "validation_sft_v2.jsonl"
SPLIT_METADATA_PATH = PROCESSED_DIR / "split_metadata_v2.json"

V3_SUMMARY_PATH = (
    PROJECT_ROOT
    / "results"
    / "minidev500_failure_mode_analysis_with_evidence_v3"
    / "failure_mode_analysis_summary_v3.json"
)

OUTPUT_DIR = PROCESSED_DIR / "targeted_sft_v1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------- Reproducible design ----------------------
SAMPLING_SEED = 20260816
TARGETED_SHARE = 0.65
GENERAL_REPLAY_SHARE = 0.35
MAX_SOURCE_OCCURRENCES = 2
MIN_UNIQUE_SOURCE_COVERAGE = 0.75
FORCE_EVIDENCE_FOR_TARGETED = False
MAX_EVIDENCE_RATE_SHIFT = 0.04

FAMILY_WEIGHTS = {
    "schema_relationship": 0.40,
    "predicate_evidence": 0.22,
    "computation_logic": 0.20,
    "result_structure": 0.18,
}

assert math.isclose(TARGETED_SHARE + GENERAL_REPLAY_SHARE, 1.0)
assert math.isclose(sum(FAMILY_WEIGHTS.values()), 1.0)

print("Output directory:", OUTPUT_DIR)
print("Family weights:", FAMILY_WEIGHTS)


Output directory: /content/drive/MyDrive/bird-text2sql-sft/processed/targeted_sft_v1
Family weights: {'schema_relationship': 0.4, 'predicate_evidence': 0.22, 'computation_logic': 0.2, 'result_structure': 0.18}


In [5]:
def load_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise RuntimeError(f"Invalid JSONL at {path}:{line_number}") from exc
    return records


def file_sha256(path):
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


required_inputs = [TRAIN_INPUT_PATH, VALIDATION_INPUT_PATH, SPLIT_METADATA_PATH]
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Run 02_data_preparation.ipynb first. Missing:\n- "
        + "\n- ".join(missing_inputs)
    )

train_records = load_jsonl(TRAIN_INPUT_PATH)
validation_records = load_jsonl(VALIDATION_INPUT_PATH)
split_metadata = json.loads(SPLIT_METADATA_PATH.read_text(encoding="utf-8"))
v3_summary = (
    json.loads(V3_SUMMARY_PATH.read_text(encoding="utf-8"))
    if V3_SUMMARY_PATH.exists()
    else None
)

print("Original train prompts:", len(train_records))
print("Original validation prompts:", len(validation_records))
print("Split method:", split_metadata.get("split_method"))
print("v3 provenance available:", v3_summary is not None)


Original train prompts: 5825
Original validation prompts: 1485
Split method: database_level_90_10
v3 provenance available: True


In [6]:
def validate_sft_record(record, location):
    required = {"source_index", "db_id", "evidence", "use_evidence", "messages"}
    missing = required - set(record)
    if missing:
        raise ValueError(f"{location}: missing fields {sorted(missing)}")
    roles = [message.get("role") for message in record["messages"]]
    if roles != ["system", "user", "assistant"]:
        raise ValueError(f"{location}: unexpected roles {roles}")
    if not str(record["messages"][-1].get("content") or "").strip():
        raise ValueError(f"{location}: missing assistant SQL")


for index, record in enumerate(train_records):
    validate_sft_record(record, f"train[{index}]")
for index, record in enumerate(validation_records):
    validate_sft_record(record, f"validation[{index}]")

train_source_ids = {int(record["source_index"]) for record in train_records}
validation_source_ids = {int(record["source_index"]) for record in validation_records}
overlap = train_source_ids & validation_source_ids

if overlap:
    raise RuntimeError(f"Train/validation source leakage detected: {len(overlap)} rows")
if len(train_source_ids) != len(train_records):
    raise RuntimeError(
        "Expected one original training prompt per source_index. "
        "Re-run notebook 02 with the v2 preparation logic."
    )
if split_metadata.get("split_method") != "database_level_90_10":
    raise RuntimeError("Expected the original database-level 90/10 split.")
if v3_summary is not None:
    pending = int(v3_summary.get("audit_counts", {}).get("pending_rows", -1))
    if pending != 0:
        raise RuntimeError(f"Failure-analysis v3 still has {pending} pending rows.")

print("Train source IDs:", len(train_source_ids))
print("Validation source IDs:", len(validation_source_ids))
print("Source overlap:", len(overlap))


Train source IDs: 5825
Validation source IDs: 776
Source overlap: 0


## SQL capability feature extraction

The selector never compares a BIRD-train row with a MiniDev prediction. It derives reproducible proxy tags only from the training row's Gold SQL, question/evidence availability, and schema text already present in its prompt.

A row can carry more than one capability tag. Multi-tag rows may appear twice, capped by `MAX_SOURCE_OCCURRENCES`, which is the mechanism that reweights the data while preserving a fixed total prompt count.


In [7]:
ARITHMETIC_TYPES = (exp.Add, exp.Sub, exp.Mul, exp.Div, exp.Mod)
PREDICATE_TYPES = (
    exp.EQ, exp.NEQ, exp.GT, exp.GTE, exp.LT, exp.LTE,
    exp.Between, exp.In, exp.Like,
)
SET_TYPES = (exp.Union, exp.Intersect, exp.Except)
DATE_STRING_TOKENS = (
    "date", "time", "year", "month", "day", "strftime",
    "substr", "substring", "trim", "lower", "upper", "replace",
    "concat", "length",
)


def assistant_sql(record):
    return str(record["messages"][-1]["content"]).strip()


def user_prompt(record):
    return str(record["messages"][1]["content"])


def function_name(node):
    if isinstance(node, exp.Anonymous):
        return str(node.name or "anonymous").lower()
    try:
        return str(node.sql_name()).lower()
    except Exception:
        return type(node).__name__.lower()


def extract_sql_features(record):
    sql = assistant_sql(record)
    try:
        tree = sqlglot.parse_one(sql, read="sqlite")
        parse_ok = True
        parse_error = ""
    except Exception as exc:
        tree = None
        parse_ok = False
        parse_error = f"{type(exc).__name__}: {exc}"

    base = {
        "source_index": int(record["source_index"]),
        "db_id": str(record["db_id"]),
        "evidence_present": bool(str(record.get("evidence") or "").strip()),
        "original_use_evidence": bool(record.get("use_evidence")),
        "schema_table_count": len(
            re.findall(r"(?i)\\bCREATE\\s+TABLE\\b", user_prompt(record))
        ),
        "sql_parse_ok": parse_ok,
        "sql_parse_error": parse_error,
    }
    if tree is None:
        base.update({
            "table_count": 0,
            "join_count": 0,
            "subquery_count": 0,
            "select_expression_count": 0,
            "aggregate_count": 0,
            "where_count": 0,
            "having_count": 0,
            "group_count": 0,
            "order_count": 0,
            "limit_count": 0,
            "distinct_count": 0,
            "set_operation_count": 0,
            "predicate_count": 0,
            "literal_count": 0,
            "arithmetic_count": 0,
            "cast_count": 0,
            "case_count": 0,
            "window_count": 0,
            "null_count": 0,
            "date_string_function_count": 0,
        })
        return base

    tables = {str(node.name).lower() for node in tree.find_all(exp.Table)}
    selects = list(tree.find_all(exp.Select))
    functions = [node for node in tree.walk() if isinstance(node, exp.Func)]
    function_names = [function_name(node) for node in functions]

    base.update({
        "table_count": len(tables),
        "join_count": len(list(tree.find_all(exp.Join))),
        "subquery_count": len(list(tree.find_all(exp.Subquery))),
        "select_expression_count": max(
            [len(select.expressions) for select in selects] or [0]
        ),
        "aggregate_count": len(list(tree.find_all(exp.AggFunc))),
        "where_count": len(list(tree.find_all(exp.Where))),
        "having_count": len(list(tree.find_all(exp.Having))),
        "group_count": len(list(tree.find_all(exp.Group))),
        "order_count": len(list(tree.find_all(exp.Order))),
        "limit_count": len(list(tree.find_all(exp.Limit))),
        "distinct_count": sum(
            select.args.get("distinct") is not None for select in selects
        ),
        "set_operation_count": sum(
            isinstance(node, SET_TYPES) for node in tree.walk()
        ),
        "predicate_count": sum(
            isinstance(node, PREDICATE_TYPES) for node in tree.walk()
        ),
        "literal_count": len(list(tree.find_all(exp.Literal))),
        "arithmetic_count": sum(
            isinstance(node, ARITHMETIC_TYPES) for node in tree.walk()
        ),
        "cast_count": len(list(tree.find_all(exp.Cast))),
        "case_count": len(list(tree.find_all(exp.Case))),
        "window_count": len(list(tree.find_all(exp.Window))),
        "null_count": len(list(tree.find_all(exp.Null))),
        "date_string_function_count": sum(
            any(token in name for token in DATE_STRING_TOKENS)
            for name in function_names
        ),
    })
    return base


train_features = pd.DataFrame(
    [extract_sql_features(record) for record in train_records]
)

parse_failures = train_features.loc[~train_features["sql_parse_ok"]]
print("Feature rows:", len(train_features))
print("Gold SQL parse failures:", len(parse_failures))
if len(parse_failures):
    display(parse_failures[["source_index", "db_id", "sql_parse_error"]].head(10))


Feature rows: 5825
Gold SQL parse failures: 0


In [8]:
FAMILY_ORDER = list(FAMILY_WEIGHTS)


def add_capability_tags(frame):
    frame = frame.copy()

    frame["tag_schema_relationship"] = (
        (frame["join_count"] > 0)
        | (frame["table_count"] >= 2)
        | ((frame["subquery_count"] > 0) & (frame["table_count"] > 0))
    )
    frame["tag_predicate_evidence"] = (
        ((frame["where_count"] + frame["having_count"]) > 0)
        & (
            (frame["predicate_count"] > 0)
            | (frame["literal_count"] > 0)
            | (frame["null_count"] > 0)
        )
    )
    frame["tag_computation_logic"] = (
        (
            frame["arithmetic_count"]
            + frame["cast_count"]
            + frame["case_count"]
            + frame["window_count"]
            + frame["null_count"]
            + frame["date_string_function_count"]
        )
        > 0
    )
    frame["tag_result_structure"] = (
        (
            frame["aggregate_count"]
            + frame["group_count"]
            + frame["having_count"]
            + frame["order_count"]
            + frame["limit_count"]
            + frame["distinct_count"]
            + frame["set_operation_count"]
        )
        > 0
    ) | (frame["select_expression_count"] >= 3)

    frame["score_schema_relationship"] = (
        1.50 * (frame["table_count"] >= 2).astype(float)
        + 0.75 * frame["join_count"].clip(upper=3)
        + 0.60 * (frame["subquery_count"] > 0).astype(float)
        + 0.25 * (frame["schema_table_count"] >= 8).astype(float)
    )
    frame["score_predicate_evidence"] = (
        0.90 * (frame["where_count"] > 0).astype(float)
        + 0.80 * (frame["having_count"] > 0).astype(float)
        + 0.35 * frame["predicate_count"].clip(upper=4)
        + 0.45 * frame["evidence_present"].astype(float)
        + 0.25 * (frame["literal_count"] >= 2).astype(float)
    )
    frame["score_computation_logic"] = (
        0.50 * frame["arithmetic_count"].clip(upper=4)
        + 0.55 * frame["cast_count"].clip(upper=2)
        + 0.75 * frame["case_count"].clip(upper=2)
        + 0.90 * frame["window_count"].clip(upper=2)
        + 0.55 * frame["null_count"].clip(upper=2)
        + 0.65 * frame["date_string_function_count"].clip(upper=3)
    )
    frame["score_result_structure"] = (
        0.55 * frame["aggregate_count"].clip(upper=4)
        + 0.75 * frame["group_count"].clip(upper=2)
        + 0.80 * frame["having_count"].clip(upper=2)
        + 0.45 * frame["order_count"].clip(upper=2)
        + 0.45 * frame["limit_count"].clip(upper=2)
        + 0.55 * frame["distinct_count"].clip(upper=2)
        + 0.90 * frame["set_operation_count"].clip(upper=2)
        + 0.20 * (frame["select_expression_count"] >= 3).astype(float)
    )

    complexity_columns = [
        "join_count", "subquery_count", "aggregate_count", "group_count",
        "having_count", "predicate_count", "arithmetic_count", "cast_count",
        "case_count", "window_count", "set_operation_count",
    ]
    frame["complexity_score"] = frame[complexity_columns].sum(axis=1)
    frame["complexity_bucket"] = pd.cut(
        frame["complexity_score"],
        bins=[-1, 3, 7, np.inf],
        labels=["simple", "moderate", "challenging"],
    ).astype(str)

    def tags_for_row(row):
        return [
            family for family in FAMILY_ORDER
            if bool(row[f"tag_{family}"])
        ]

    frame["capability_tags"] = frame.apply(tags_for_row, axis=1)
    frame["capability_tag_count"] = frame["capability_tags"].map(len)
    return frame


train_features = add_capability_tags(train_features)
candidate_summary = pd.DataFrame([
    {
        "family": family,
        "candidate_rows": int(train_features[f"tag_{family}"].sum()),
        "candidate_share": float(train_features[f"tag_{family}"].mean()),
        "target_weight": FAMILY_WEIGHTS[family],
    }
    for family in FAMILY_ORDER
])
display(candidate_summary.style.format({
    "candidate_share": "{:.1%}",
    "target_weight": "{:.1%}",
}))
print("Rows with at least one target tag:", int((train_features.capability_tag_count > 0).sum()))


,family,candidate_rows,candidate_share,target_weight
0,schema_relationship,4570,78.5%,40.0%
1,predicate_evidence,5199,89.3%,22.0%
2,computation_logic,1257,21.6%,20.0%
3,result_structure,3966,68.1%,18.0%


Rows with at least one target tag: 5824


## Deterministic stratified resampling

Sampling uses inverse-square-root database frequency and a moderate/challenging complexity boost. This prevents the largest training databases and easiest structures from dominating. Each source row can appear at most twice across all targeted and replay selections.


In [9]:
def allocate_integer_quotas(total, weights):
    raw = {key: total * value for key, value in weights.items()}
    quotas = {key: int(math.floor(value)) for key, value in raw.items()}
    remainder = total - sum(quotas.values())
    order = sorted(raw, key=lambda key: raw[key] - quotas[key], reverse=True)
    for key in order[:remainder]:
        quotas[key] += 1
    return quotas


def sample_from_slots(candidate_indices, quota, per_index_weight, occurrences, rng):
    slots = []
    weights = []
    for index in candidate_indices:
        remaining_capacity = MAX_SOURCE_OCCURRENCES - occurrences[index]
        if remaining_capacity <= 0:
            continue
        for _ in range(remaining_capacity):
            slots.append(index)
            weights.append(max(float(per_index_weight[index]), 1e-12))

    if len(slots) < quota:
        raise RuntimeError(
            f"Only {len(slots)} source-capacity slots are available for quota {quota}. "
            "Lower TARGETED_SHARE or raise MAX_SOURCE_OCCURRENCES."
        )

    probabilities = np.asarray(weights, dtype=float)
    probabilities = probabilities / probabilities.sum()
    selected_slot_positions = rng.choice(
        len(slots), size=quota, replace=False, p=probabilities
    )
    selected = [slots[position] for position in selected_slot_positions]
    for index in selected:
        occurrences[index] += 1
    return selected


rng = np.random.default_rng(SAMPLING_SEED)
output_size = len(train_records)
targeted_count = int(round(output_size * TARGETED_SHARE))
replay_count = output_size - targeted_count
family_quotas = allocate_integer_quotas(targeted_count, FAMILY_WEIGHTS)

db_counts = train_features["db_id"].value_counts().to_dict()
inverse_db_weight = train_features["db_id"].map(
    lambda db_id: 1.0 / math.sqrt(db_counts[db_id])
).astype(float)
complexity_weight = train_features["complexity_bucket"].map({
    "simple": 0.80,
    "moderate": 1.25,
    "challenging": 1.45,
}).astype(float)

occurrences = Counter()
selections = []

# Protect scarce capability pools from having all of their source capacity
# consumed by broader, overlapping families.
family_sampling_order = sorted(
    FAMILY_ORDER,
    key=lambda family: (
        int(train_features[f"tag_{family}"].sum())
        * MAX_SOURCE_OCCURRENCES
        / max(family_quotas[family], 1)
    ),
)
print("Family sampling order (scarcest first):", family_sampling_order)

for family in family_sampling_order:
    candidates = train_features.index[train_features[f"tag_{family}"]].tolist()
    family_weight = (
        (train_features[f"score_{family}"] + 0.25)
        * inverse_db_weight
        * complexity_weight
    ).to_dict()
    chosen = sample_from_slots(
        candidates,
        family_quotas[family],
        family_weight,
        occurrences,
        rng,
    )
    selections.extend({
        "record_index": int(index),
        "selection_role": "targeted",
        "target_family": family,
    } for index in chosen)

# Replay is drawn independently, with a strong bonus for sources not yet selected.
replay_candidates = train_features.index.tolist()
replay_weight = {
    int(index): float(inverse_db_weight.loc[index])
    * (3.0 if occurrences[index] == 0 else 0.55)
    for index in replay_candidates
}
replay_chosen = sample_from_slots(
    replay_candidates,
    replay_count,
    replay_weight,
    occurrences,
    rng,
)
selections.extend({
    "record_index": int(index),
    "selection_role": "general_replay",
    "target_family": "general_replay",
} for index in replay_chosen)

rng.shuffle(selections)

selected_source_ids = {
    int(train_records[item["record_index"]]["source_index"])
    for item in selections
}
unique_coverage = len(selected_source_ids) / len(train_source_ids)

print("Output prompts:", len(selections))
print("Targeted prompts:", targeted_count)
print("Replay prompts:", replay_count)
print("Family quotas:", family_quotas)
print(f"Unique source coverage: {unique_coverage:.1%}")
print("Maximum source occurrences:", max(occurrences.values()))

assert len(selections) == output_size
assert sum(item["selection_role"] == "targeted" for item in selections) == targeted_count
assert max(occurrences.values()) <= MAX_SOURCE_OCCURRENCES
assert unique_coverage >= MIN_UNIQUE_SOURCE_COVERAGE


Family sampling order (scarcest first): ['computation_logic', 'schema_relationship', 'result_structure', 'predicate_evidence']
Output prompts: 5825
Targeted prompts: 3786
Replay prompts: 2039
Family quotas: {'schema_relationship': 1514, 'predicate_evidence': 833, 'computation_logic': 757, 'result_structure': 682}
Unique source coverage: 77.8%
Maximum source occurrences: 2


In [10]:
EVIDENCE_PATTERN = re.compile(
    r"(External knowledge:\n)(.*?)(\n\nQuestion:\n)",
    flags=re.DOTALL,
)


def set_evidence_condition(record, use_evidence):
    result = deepcopy(record)
    evidence = str(result.get("evidence") or "").strip()
    effective_use_evidence = bool(use_evidence and evidence)
    evidence_block = evidence if effective_use_evidence else "None"

    original_prompt = str(result["messages"][1]["content"])
    updated_prompt, replacement_count = EVIDENCE_PATTERN.subn(
        lambda match: match.group(1) + evidence_block + match.group(3),
        original_prompt,
        count=1,
    )
    if replacement_count != 1:
        raise RuntimeError(
            "Could not locate the External knowledge block in source_index="
            f"{result['source_index']}"
        )

    result["messages"][1]["content"] = updated_prompt
    result["use_evidence"] = effective_use_evidence
    return result


targeted_train_records = []
selection_rows = []

for sample_number, item in enumerate(selections):
    record_index = item["record_index"]
    original = train_records[record_index]
    feature = train_features.loc[record_index]
    role = item["selection_role"]
    family = item["target_family"]

    if role == "targeted" and FORCE_EVIDENCE_FOR_TARGETED:
        output_record = set_evidence_condition(original, use_evidence=True)
    else:
        output_record = deepcopy(original)

    sample_instance_id = f"tsft-v1-{sample_number:06d}"
    output_record.update({
        "sample_instance_id": sample_instance_id,
        "selection_role": role,
        "target_family": family,
        "all_target_families": list(feature["capability_tags"]),
        "complexity_bucket": str(feature["complexity_bucket"]),
        "targeted_sampling_version": "bird_targeted_sft_v1",
        "targeted_sampling_seed": SAMPLING_SEED,
        "original_use_evidence": bool(original.get("use_evidence")),
    })
    targeted_train_records.append(output_record)

    selection_rows.append({
        "sample_instance_id": sample_instance_id,
        "source_index": int(original["source_index"]),
        "db_id": str(original["db_id"]),
        "selection_role": role,
        "target_family": family,
        "all_target_families": json.dumps(
            list(feature["capability_tags"]), ensure_ascii=False
        ),
        "complexity_bucket": str(feature["complexity_bucket"]),
        "complexity_score": float(feature["complexity_score"]),
        "original_use_evidence": bool(original.get("use_evidence")),
        "output_use_evidence": bool(output_record.get("use_evidence")),
        "evidence_present": bool(feature["evidence_present"]),
        "sql_parse_ok": bool(feature["sql_parse_ok"]),
        "table_count": int(feature["table_count"]),
        "join_count": int(feature["join_count"]),
        "predicate_count": int(feature["predicate_count"]),
        "aggregate_count": int(feature["aggregate_count"]),
        "arithmetic_count": int(feature["arithmetic_count"]),
    })

selection_index = pd.DataFrame(selection_rows)
original_train_evidence_rate = float(np.mean([
    bool(record.get("use_evidence")) for record in train_records
]))
output_train_evidence_rate = float(selection_index.output_use_evidence.mean())
print("Targeted-train evidence rate:", f"{selection_index.output_use_evidence.mean():.1%}")
print("Original full-train evidence rate:", f"{original_train_evidence_rate:.1%}")
print("Evidence-rate shift:", f"{output_train_evidence_rate - original_train_evidence_rate:+.1%}")
if not FORCE_EVIDENCE_FOR_TARGETED:
    assert abs(output_train_evidence_rate - original_train_evidence_rate) <= MAX_EVIDENCE_RATE_SHIFT
display(pd.crosstab(
    selection_index.selection_role,
    selection_index.output_use_evidence,
    margins=True,
))


Targeted-train evidence rate: 50.9%
Original full-train evidence rate: 50.0%
Evidence-rate shift: +0.9%


output_use_evidence,False,True,All
selection_role,,,
general_replay,1047,992,2039
targeted,1812,1974,3786
All,2859,2966,5825


In [11]:
# Preserve the complete general validation set for checkpoint selection.
validation_general_records = deepcopy(validation_records)

validation_feature_rows = [extract_sql_features(record) for record in validation_records]
validation_features = add_capability_tags(pd.DataFrame(validation_feature_rows))

# Select one evidence-preferred record per validation source for targeted diagnostics.
validation_candidates = []
for record_index, record in enumerate(validation_records):
    feature = validation_features.loc[record_index]
    if not feature["capability_tags"]:
        continue
    family = max(
        feature["capability_tags"],
        key=lambda name: float(feature[f"score_{name}"]) * FAMILY_WEIGHTS[name],
    )
    validation_candidates.append({
        "record_index": record_index,
        "source_index": int(record["source_index"]),
        "prefer_score": int(bool(record.get("use_evidence"))) * 10
            + float(feature[f"score_{family}"]),
        "target_family": family,
        "all_target_families": list(feature["capability_tags"]),
        "complexity_bucket": str(feature["complexity_bucket"]),
    })

validation_candidate_df = pd.DataFrame(validation_candidates).sort_values(
    ["source_index", "prefer_score"], ascending=[True, False]
)
validation_selected = validation_candidate_df.drop_duplicates(
    "source_index", keep="first"
)

validation_targeted_records = []
validation_index_rows = []
for _, item in validation_selected.iterrows():
    record = deepcopy(validation_records[int(item.record_index)])
    record.update({
        "selection_role": "targeted_validation",
        "target_family": item.target_family,
        "all_target_families": item.all_target_families,
        "complexity_bucket": item.complexity_bucket,
        "targeted_sampling_version": "bird_targeted_sft_v1",
    })
    validation_targeted_records.append(record)
    validation_index_rows.append({
        "source_index": int(record["source_index"]),
        "db_id": str(record["db_id"]),
        "target_family": item.target_family,
        "all_target_families": json.dumps(
            item.all_target_families, ensure_ascii=False
        ),
        "complexity_bucket": item.complexity_bucket,
        "use_evidence": bool(record.get("use_evidence")),
    })

validation_targeted_index = pd.DataFrame(validation_index_rows)

print("General validation prompts:", len(validation_general_records))
print("Targeted diagnostic validation prompts:", len(validation_targeted_records))
display(validation_targeted_index.target_family.value_counts().rename("count").to_frame())


General validation prompts: 1485
Targeted diagnostic validation prompts: 775


,count
target_family,
schema_relationship,629
predicate_evidence,117
result_structure,21
computation_logic,8


In [12]:
# -------------------------- Integrity gates --------------------------
for index, record in enumerate(targeted_train_records):
    validate_sft_record(record, f"targeted_train[{index}]")
for index, record in enumerate(validation_general_records):
    validate_sft_record(record, f"validation_general[{index}]")
for index, record in enumerate(validation_targeted_records):
    validate_sft_record(record, f"validation_targeted[{index}]")

output_train_source_ids = {
    int(record["source_index"]) for record in targeted_train_records
}
output_validation_source_ids = {
    int(record["source_index"]) for record in validation_general_records
}

assert len(targeted_train_records) == len(train_records)
assert not (output_train_source_ids & output_validation_source_ids)
assert output_train_source_ids <= train_source_ids
assert output_validation_source_ids == validation_source_ids
assert all(
    record["messages"][-1]["content"].strip()
    == train_records[item["record_index"]]["messages"][-1]["content"].strip()
    for record, item in zip(targeted_train_records, selections)
)
assert all(
    record["messages"][-1]["content"].strip()
    == original["messages"][-1]["content"].strip()
    for record, original in zip(validation_general_records, validation_records)
)

print("All leakage, size, message-role, and Gold-SQL preservation checks passed.")


All leakage, size, message-role, and Gold-SQL preservation checks passed.


In [13]:
family_summary = (
    selection_index.groupby(["selection_role", "target_family"], dropna=False)
    .agg(
        prompt_count=("sample_instance_id", "size"),
        unique_sources=("source_index", "nunique"),
        unique_databases=("db_id", "nunique"),
        evidence_rate=("output_use_evidence", "mean"),
        mean_complexity=("complexity_score", "mean"),
    )
    .reset_index()
)
family_summary["share_of_train"] = (
    family_summary.prompt_count / len(selection_index)
)

database_summary = (
    selection_index.groupby(["db_id", "selection_role"])
    .agg(
        prompt_count=("sample_instance_id", "size"),
        unique_sources=("source_index", "nunique"),
        evidence_rate=("output_use_evidence", "mean"),
    )
    .reset_index()
)

complexity_summary = (
    selection_index.groupby(["selection_role", "complexity_bucket"])
    .size()
    .rename("prompt_count")
    .reset_index()
)

display(family_summary.style.format({
    "evidence_rate": "{:.1%}",
    "share_of_train": "{:.1%}",
    "mean_complexity": "{:.2f}",
}))
display(complexity_summary)


,selection_role,target_family,prompt_count,unique_sources,unique_databases,evidence_rate,mean_complexity,share_of_train
0,general_replay,general_replay,2039,1705,60,48.7%,3.69,35.0%
1,targeted,computation_logic,757,599,61,54.4%,8.48,13.0%
2,targeted,predicate_evidence,833,793,62,52.1%,4.82,14.3%
3,targeted,result_structure,682,637,62,52.8%,5.30,11.7%
4,targeted,schema_relationship,1514,1390,62,50.7%,5.51,26.0%


,selection_role,complexity_bucket,prompt_count
0,general_replay,challenging,106
1,general_replay,moderate,773
2,general_replay,simple,1160
3,targeted,challenging,867
4,targeted,moderate,2207
5,targeted,simple,712


In [14]:
TRAIN_OUTPUT_PATH = OUTPUT_DIR / "train_targeted_sft_v1.jsonl"
VALIDATION_GENERAL_OUTPUT_PATH = OUTPUT_DIR / "validation_general_sft_v1.jsonl"
VALIDATION_TARGETED_OUTPUT_PATH = OUTPUT_DIR / "validation_targeted_sft_v1.jsonl"
SELECTION_INDEX_PATH = OUTPUT_DIR / "train_selection_index_v1.csv"
VALIDATION_INDEX_PATH = OUTPUT_DIR / "validation_targeted_index_v1.csv"
FAMILY_SUMMARY_PATH = OUTPUT_DIR / "targeted_family_summary_v1.csv"
DATABASE_SUMMARY_PATH = OUTPUT_DIR / "targeted_database_summary_v1.csv"
COMPLEXITY_SUMMARY_PATH = OUTPUT_DIR / "targeted_complexity_summary_v1.csv"
MANIFEST_PATH = OUTPUT_DIR / "targeted_sft_manifest_v1.json"


def atomic_write_jsonl(records, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        mode="w",
        encoding="utf-8",
        dir=output_path.parent,
        prefix=output_path.name + ".",
        suffix=".tmp",
        delete=False,
    ) as handle:
        temporary = Path(handle.name)
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    os.replace(temporary, output_path)


def atomic_write_csv(frame, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_path.with_suffix(output_path.suffix + ".tmp")
    frame.to_csv(temporary, index=False)
    os.replace(temporary, output_path)


atomic_write_jsonl(targeted_train_records, TRAIN_OUTPUT_PATH)
atomic_write_jsonl(validation_general_records, VALIDATION_GENERAL_OUTPUT_PATH)
atomic_write_jsonl(validation_targeted_records, VALIDATION_TARGETED_OUTPUT_PATH)
atomic_write_csv(selection_index, SELECTION_INDEX_PATH)
atomic_write_csv(validation_targeted_index, VALIDATION_INDEX_PATH)
atomic_write_csv(family_summary, FAMILY_SUMMARY_PATH)
atomic_write_csv(database_summary, DATABASE_SUMMARY_PATH)
atomic_write_csv(complexity_summary, COMPLEXITY_SUMMARY_PATH)

manifest = {
    "version": "bird_targeted_sft_v1",
    "created_with": "BIRD_Targeted_SFT_Data_Construction_v1.ipynb",
    "data_source_rule": (
        "Outputs contain BIRD train-split records only. MiniDev500 contributed "
        "aggregate capability weights but no training rows."
    ),
    "inputs": {
        "train_path": str(TRAIN_INPUT_PATH),
        "train_sha256": file_sha256(TRAIN_INPUT_PATH),
        "validation_path": str(VALIDATION_INPUT_PATH),
        "validation_sha256": file_sha256(VALIDATION_INPUT_PATH),
        "split_metadata_path": str(SPLIT_METADATA_PATH),
        "split_metadata_sha256": file_sha256(SPLIT_METADATA_PATH),
        "v3_summary_path": str(V3_SUMMARY_PATH) if V3_SUMMARY_PATH.exists() else None,
        "v3_summary_sha256": file_sha256(V3_SUMMARY_PATH) if V3_SUMMARY_PATH.exists() else None,
    },
    "sampling": {
        "seed": SAMPLING_SEED,
        "output_train_prompts": len(targeted_train_records),
        "targeted_share": TARGETED_SHARE,
        "general_replay_share": GENERAL_REPLAY_SHARE,
        "family_weights": FAMILY_WEIGHTS,
        "family_quotas": family_quotas,
        "max_source_occurrences": MAX_SOURCE_OCCURRENCES,
        "unique_source_count": len(output_train_source_ids),
        "unique_source_coverage": unique_coverage,
        "force_evidence_for_targeted": FORCE_EVIDENCE_FOR_TARGETED,
        "original_train_evidence_rate": original_train_evidence_rate,
        "output_evidence_rate": output_train_evidence_rate,
        "maximum_allowed_evidence_rate_shift": MAX_EVIDENCE_RATE_SHIFT,
    },
    "split_integrity": {
        "split_method": split_metadata.get("split_method"),
        "train_source_count_original": len(train_source_ids),
        "validation_source_count_original": len(validation_source_ids),
        "train_validation_source_overlap": 0,
        "training_database_ids": split_metadata.get("training_database_ids", []),
        "validation_database_ids": split_metadata.get("validation_database_ids", []),
    },
    "outputs": {},
}

output_paths = [
    TRAIN_OUTPUT_PATH,
    VALIDATION_GENERAL_OUTPUT_PATH,
    VALIDATION_TARGETED_OUTPUT_PATH,
    SELECTION_INDEX_PATH,
    VALIDATION_INDEX_PATH,
    FAMILY_SUMMARY_PATH,
    DATABASE_SUMMARY_PATH,
    COMPLEXITY_SUMMARY_PATH,
]
manifest["outputs"] = {
    path.name: {
        "path": str(path),
        "sha256": file_sha256(path),
        "size_bytes": path.stat().st_size,
    }
    for path in output_paths
}

temporary_manifest = MANIFEST_PATH.with_suffix(".json.tmp")
temporary_manifest.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
os.replace(temporary_manifest, MANIFEST_PATH)

print("Saved Targeted-SFT artifacts:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f" - {path.name}: {path.stat().st_size / 1024:.1f} KB")


Saved Targeted-SFT artifacts:
 - targeted_complexity_summary_v1.csv: 0.2 KB
 - targeted_database_summary_v1.csv: 5.6 KB
 - targeted_family_summary_v1.csv: 0.6 KB
 - targeted_sft_manifest_v1.json: 5.7 KB
 - train_selection_index_v1.csv: 981.4 KB
 - train_targeted_sft_v1.jsonl: 34024.7 KB
 - validation_general_sft_v1.jsonl: 6544.4 KB
 - validation_targeted_index_v1.csv: 84.9 KB
 - validation_targeted_sft_v1.jsonl: 3639.4 KB


In [15]:
# Reload from disk and verify counts/hashes after serialization.
reloaded_train = load_jsonl(TRAIN_OUTPUT_PATH)
reloaded_validation_general = load_jsonl(VALIDATION_GENERAL_OUTPUT_PATH)
reloaded_validation_targeted = load_jsonl(VALIDATION_TARGETED_OUTPUT_PATH)
reloaded_manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

assert len(reloaded_train) == len(targeted_train_records)
assert len(reloaded_validation_general) == len(validation_general_records)
assert len(reloaded_validation_targeted) == len(validation_targeted_records)
assert reloaded_manifest["sampling"]["output_train_prompts"] == len(reloaded_train)
assert file_sha256(TRAIN_OUTPUT_PATH) == reloaded_manifest["outputs"][TRAIN_OUTPUT_PATH.name]["sha256"]

print("Reload verification passed.")
print("Training file:", TRAIN_OUTPUT_PATH)
print("Checkpoint-selection validation:", VALIDATION_GENERAL_OUTPUT_PATH)
print("Targeted diagnostic validation:", VALIDATION_TARGETED_OUTPUT_PATH)


Reload verification passed.
Training file: /content/drive/MyDrive/bird-text2sql-sft/processed/targeted_sft_v1/train_targeted_sft_v1.jsonl
Checkpoint-selection validation: /content/drive/MyDrive/bird-text2sql-sft/processed/targeted_sft_v1/validation_general_sft_v1.jsonl
Targeted diagnostic validation: /content/drive/MyDrive/bird-text2sql-sft/processed/targeted_sft_v1/validation_targeted_sft_v1.jsonl


## Next experiment

Train a **new LoRA adapter initialized from the same original Qwen2.5-Coder-7B-Instruct base**, not by continuing the old adapter.

Controlled comparison:

1. Keep tokenizer, prompt template, LoRA rank/alpha/dropout, optimizer, learning rate, seed, max length, decoding parameters, and total optimizer steps fixed.
2. Replace only the training JSONL with `train_targeted_sft_v1.jsonl`.
3. Use `validation_general_sft_v1.jsonl` for `load_best_model_at_end`; evaluate `validation_targeted_sft_v1.jsonl` as a secondary diagnostic.
4. Evaluate Base + evidence, Original SFT + evidence, and Targeted SFT + evidence on MiniDev500.
5. Report EX@1, Oracle@3/5, valid-SQL rate, simple/moderate/challenging EX, paired improvements/regressions, and McNemar or paired-bootstrap uncertainty.
6. Treat MiniDev500 as development data. Lock a separate untouched BIRD dev subset before further tuning for final reporting.

Suggested ablations after the first controlled run:

- Targeted-SFT with forced evidence on targeted copies (`FORCE_EVIDENCE_FOR_TARGETED=True`).
- 50/50 instead of 65/35 targeted/replay.
- Reranking only on Oracle@5 selection-opportunity cases.
